In [0]:
/Volumes/workspace/gold/exporttables = ["dim_store", "dim_customer", "dim_product", "dim_date",
          "fact_sales", "fact_inventory_snapshot"]

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.gold.export")
out = "/Volumes/workspace/gold/export"

for t in tables:
    tmp, dest = f"{out}/_tmp_{t}", f"{out}/{t}"
    df = spark.table(f"workspace.gold.{t}")

    # Spark writes one part file inside a folder
    df.coalesce(1).write.mode("overwrite").parquet(tmp)

    # Copy that part file out as a single, properly named file
    part = [f.path for f in dbutils.fs.ls(tmp) if f.name.endswith(".parquet")][0]
    dbutils.fs.rm(dest, True)
    dbutils.fs.mkdirs(dest)
    dbutils.fs.cp(part, f"{dest}/{t}.parquet")
    dbutils.fs.rm(tmp, True)

    print(f"{t}: {df.count()} rows -> {dest}/{t}.parquet")

dim_store: 10 rows -> /Volumes/workspace/gold/export/dim_store/dim_store.parquet
dim_customer: 200 rows -> /Volumes/workspace/gold/export/dim_customer/dim_customer.parquet
dim_product: 20 rows -> /Volumes/workspace/gold/export/dim_product/dim_product.parquet
dim_date: 88 rows -> /Volumes/workspace/gold/export/dim_date/dim_date.parquet
fact_sales: 12001 rows -> /Volumes/workspace/gold/export/fact_sales/fact_sales.parquet
fact_inventory_snapshot: 1400 rows -> /Volumes/workspace/gold/export/fact_inventory_snapshot/fact_inventory_snapshot.parquet


In [0]:
%sql 

SELECT s.store_id, s.store_name,
       COUNT(f.order_item_id)     AS lines,
       COUNT(DISTINCT f.order_id) AS orders,
       SUM(f.net_revenue)         AS net_revenue
FROM workspace.gold.dim_store s
LEFT JOIN workspace.gold.fact_sales f ON f.store_id = s.store_id
GROUP BY s.store_id, s.store_name
ORDER BY s.store_id;

store_id,store_name,lines,orders,net_revenue
1,RetailMart Noida Sec 18,1201,400,0.00
2,RetailMart Gurugram Cyber Hub,1201,400,882264.00
3,RetailMart Lucknow Hazratganj,1201,400,1604236.00
4,RetailMart Delhi CP,1198,400,1455990.00
5,RetailMart Kanpur Mall Road,1201,400,1984995.00
6,RetailMart Mumbai Andheri,1201,400,1701480.00
7,RetailMart Pune Baner,1198,400,2108078.00
8,RetailMart Bengaluru Indiranagar,1201,400,1599446.00
9,RetailMart Hyderabad Hitech City,1201,400,1638510.00
10,RetailMart Jaipur MI Road,1198,400,1136080.00


In [0]:
%sql
SELECT s.store_name, f.order_status,
       COUNT(DISTINCT f.order_id) AS orders,
       SUM(f.line_total)          AS gross_sales
FROM workspace.gold.fact_sales f
JOIN workspace.gold.dim_store s ON s.store_id = f.store_id
WHERE f.store_id = 1
GROUP BY s.store_name, f.order_status;

store_name,order_status,orders,gross_sales
RetailMart Noida Sec 18,Cancelled,400,1416645.00
